## Test the greedy layout design strategy

In [ ]:
import os
import sys
sys.path.append('..')
import logging
logging.basicConfig(level=logging.WARN)

from qunoc.utils.utils import get_timestamp
from qunoc.element.package import *

result_dir = f"result/layout/greedy/example_{get_timestamp()}"
os.makedirs(result_dir, exist_ok=True)

package = Package((50, 50), FabricateModel(20, 10), design_strategy="greedy", result_dir=result_dir)
# package = Package(design_strategy="greedy", result_dir=result_dir)

package.save_design_history()
package.save()
print("Package score:", package.estimate_score())

Initial score:  33.878021187989866
New action:  (ComputeModule_1_1_48_48, Strip, Y, 4)
New score:  80.83825999419155
New action:  (ComputeModule_31_1_34_48, Strip, X, 20)
New score:  104.43789658919748
Package score:  86.76993159278022


In [1]:
import os
import sys
sys.path.append('..')
import logging
logging.basicConfig(level=logging.INFO)

from qunoc.element.package import *

result_dir = f"../evaluate/result/layout_scalability/package_size/100_100_greedy"
package = Package(pre_designed_package_path=os.path.join(result_dir, "package.pkl"))
print("Package score:", package.estimate_score())

INFO:root:Package loaded from ../evaluate/result/layout_scalability/package_size/100_100_greedy/package.pkl


Package score: 16.420669420441193


## Test the rl-based layout design strategy

In [ ]:
import os
import sys
sys.path.append('..')
import logging
logging.basicConfig(level=logging.WARN)

from qunoc.utils.utils import get_timestamp
from qunoc.element.package import *
from qunoc.layout.env import QuNoCEnv
from qunoc.layout.agent import LayoutAgent

result_dir = f"result/layout/rl/example_{get_timestamp()}"
os.makedirs(result_dir, exist_ok=True)

env = QuNoCEnv((50, 50), FabricateModel(20, 10), result_dir=result_dir)
# env = QuNoCEnv(result_dir=result_dir)
agent = LayoutAgent(env, result_dir=result_dir)
agent.train(1000)
package = agent.design()

package.save_design_history()
package.save()
print("Package score:", package.estimate_score())

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Training agent...
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 16.9     |
|    ep_rew_mean     | -1.4e+09 |
| time/              |          |
|    fps             | 8        |
|    iterations      | 1        |
|    time_elapsed    | 249      |
|    total_timesteps | 2048     |
---------------------------------
Designing layout...
Initial score:  39.61728499372216
New action:  [17 40  2  1  1]
New score:  74.54653705609819
New action:  [28 23  1  0  3]
New score:  65.51810600817757
New action:  [28 23  1  0  3]
New score:  71.03631243430449
New action:  [28 23  1  0  3]
New score:  70.1210073894442
New action:  [28 23  1  0  3]
New score:  80.50475805612783
New action:  [28 23  1  0  3]
New score:  70.70797684069667
New action:  [28 23  1  0  3]
New score:  68.75900348327541
New action:  [28 23  1  0  3]
New score:  67.74360363335184
New action:  [2

## Test the benchmark generation

In [1]:
import os
import sys
sys.path.append('..')
import logging
logging.basicConfig(level=logging.WARN)

from qunoc.element.benchmark import *

vqe_program = vqe(5)
print(vqe_program)
qft_program = qft(5)
print(qft_program)
qaoa_program = qaoa(5)
print(qaoa_program)
bv_program = bv(5)
print(bv_program)

vqe program: Control_0_1, Control_0_2, Control_0_3, Control_0_4, Control_1_2, Control_1_3, Control_1_4, Control_2_3, Control_2_4, Control_3_4
qft program: Control_1_0, Control_2_0, Control_3_0, Control_2_1, Control_4_0, Control_3_1, Control_4_1, Control_3_2, Control_4_2, Control_4_3
qaoa program: Control_2_4, Control_3_4, Control_0_2, Control_1_4, Control_1_2
bv program: Control_0_3, Control_0_4


## Test the routing algorithm

In [ ]:
import os
import sys
sys.path.append('..')

from qunoc.utils.utils import get_timestamp
from qunoc.element.package import *
from qunoc.element.benchmark import *

algo = "vqe"
data_qubit_num = 16

# result_dir = f"result/routing/example_{get_timestamp()}"
result_dir = f"result/routing/example_{algo}_{data_qubit_num}"
os.makedirs(result_dir, exist_ok=True)

import logging
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(
    filename=os.path.join(result_dir, "log.txt"),
    filemode='w',
    format='%(asctime)s - %(levelname)s - %(message)s',
    level=logging.INFO,
    encoding='utf-8'
)

package = Package(package_size=(9, 9), fab_model=FabricateModel(9, 9), design_strategy="manual", result_dir=result_dir)
package.design_module_top_down(package.compute_modules[0], DesignTemplate.Grid, DesignOrientation.X, 3)

package.save_design_history()
package.save()
print("Package score:", package.estimate_score())

program = get_program(algo, data_qubit_num)
gate_depend_fig = program.draw_depend()
gate_depend_fig.savefig(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_gate_depend.png"), bbox_inches='tight')
qiskit_circuit_fig = program.draw_circuit()
qiskit_circuit_fig.savefig(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_qiskit_circuit.png"), bbox_inches='tight')
print("Program gate num:", program.gate_depend_graph.number_of_nodes())

Package score: 46.10721452574779
Program gate num: 106


In [2]:
from qunoc.utils.config import config
from qunoc.routing.router import *

config.debug_mapping = True
config.debug_block = True
config.debug_action = True
router = Router(package=package, program=program, result_dir=result_dir)
router.map_qubits()
print(f"Routing program {program.algo_name}-{program.data_qubit_num}")

Routing program vqe-15


In [ ]:
router.compile_blocks_and_actions()
router.save_block_history()
block_depend_fig = router.draw_block_depend()
block_depend_fig.savefig(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_block_depend.png"), bbox_inches='tight')
print("Block depth:", router.get_block_depth(), ", block count:", router.block_depend_graph.number_of_nodes())

Block depth: 13 , block count: 21


In [ ]:
try:
    router.schedule_actions_to_circuit()
except AssertionError:
    router.save_action_history()
    raise
router.save_action_history()
action_depend_fig = router.draw_action_depend()
action_depend_fig.savefig(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_action_depend.png"), bbox_inches='tight')
print("Action depth:", router.get_action_depth(), ", action count:", router.action_depend_graph.number_of_nodes())
circuit = router.circuit
circuit.to_pickle(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_qunoc_circuit.pkl"))
print("QuNoC gate fidelity:", router.estimate_average_gate_fide(), ", QuNoC circuit depth:", router.estimate_depth())

Action depth: 137 , action count: 711
QuNoC gate fidelity: 0.8496362314382685 , QuNoC circuit depth: 44734.0


# Test the baseline MECH

In [1]:
import os
import sys
sys.path.append('..')

from baseline.MECH.MECHBenchmarks import *
import json
import os
%load_ext autoreload
%autoreload 2

directory_path = "exp_data"
if not os.path.exists(directory_path):
    os.makedirs(directory_path)

In [4]:
structure='square'
chiplet_array_dim=(3,3)
chiplet_size=(10,10)
benchmarks=['qaoa']

exp_results = MECH_experiments(structure=structure, chiplet_array_dim=chiplet_array_dim, chiplet_size=chiplet_size, benchmarks=benchmarks, cross_link_sparsity=1)
print(exp_results)

160200/320400 gates are selected: [(0, 2), (0, 3), (0, 4), (0, 6), (0, 7), (0, 10), (0, 11), (0, 14), (0, 17), (0, 21)]...[(794, 797), (794, 799), (794, 800), (795, 799), (795, 800), (796, 797), (796, 799), (797, 798), (797, 800), (799, 800)]
----------------executing  qaoa
0/799
10/799
20/799
30/799
40/799
50/799
60/799
70/799
80/799
90/799
100/799
110/799
120/799
130/799
140/799
150/799
160/799
170/799
180/799
190/799
200/799
210/799
220/799
230/799
240/799
250/799
260/799
270/799
280/799
290/799
300/799
310/799
320/799
330/799
340/799
350/799
360/799
370/799
380/799
390/799
400/799
410/799
420/799
430/799
440/799
450/799
460/799
470/799
480/799
490/799
500/799
510/799
520/799
530/799
540/799
550/799
560/799
570/799
580/799
590/799
600/799
610/799
620/799
630/799
640/799
650/799
660/799
670/799
680/799
690/799
700/799
710/799
720/799
730/799
740/799
750/799
760/799
770/799
780/799
790/799


AssertionError: idx = -1 should >= 0

In [ ]:
structure_abbr = '_'.join([s[:3] for s in structure.split('_')])
file_path = "./result/baseline/mech/{}{}{}{}{}.json".format(structure_abbr, chiplet_array_dim[0], chiplet_array_dim[1], chiplet_size[0], chiplet_size[1])
os.makedirs(os.path.dirname(file_path), exist_ok=True)

with open(file_path, 'w') as json_file:
    json.dump(exp_results, json_file)

print(f"The dictionary has been written to {file_path}")

The dictionary has been written to ./result/baseline/mech/squ2233.json


In [ ]:
from baseline.utils.utils import get_mech_circuit_depth, get_mech_gate_fidelity

for b in benchmarks:
    mech_gate_fidelity = get_mech_gate_fidelity(exp_results[b], chiplet_size=chiplet_size)
    mech_circuit_depth = get_mech_circuit_depth(exp_results[b])
    print("Program", b+'-'+str(exp_results[b]['data_qubit_num']), ", gate num:", exp_results[b]['gate_num'], ", MECH gate fidelity:", mech_gate_fidelity, ", MECH circuit depth:", mech_circuit_depth)

Program vqe-16 , MECH gate fidelity: 0.9833966317417787 , MECH circuit depth: 574560
